In [70]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.evaluation import RegressionEvaluator
from datetime import datetime

print("🚀 Démarrage Monitoring ML")


🚀 Démarrage Monitoring ML


In [73]:
predictions = spark.table("iceberg.gold.trip_duration_predictions")
spark.table("iceberg.gold.trip_duration_predictions").limit(5).toPandas()


,tpep_pickup_datetime,pickup_zone,pickup_borough,dropoff_zone,trip_duration_minutes,predicted_duration,prediction_error,abs_error,is_peak_hour,model_version
0,2025-09-01 09:17:56,Bensonhurst West,Brooklyn,Homecrest,51.633333,27.949837,-23.683497,23.683497,True,rf_v1
1,2025-09-01 10:26:21,Bensonhurst West,Brooklyn,Brighton Beach,49.550000,34.392839,-15.157161,15.157161,False,rf_v1
2,2025-09-01 14:24:32,Bensonhurst West,Brooklyn,Starrett City,32.533333,40.658953,8.125620,8.125620,False,rf_v1
3,2025-09-01 17:47:27,Bensonhurst West,Brooklyn,Canarsie,27.083333,28.618943,1.535609,1.535609,True,rf_v1
4,2025-09-02 06:35:14,Bensonhurst West,Brooklyn,Union Sq,51.050000,67.530952,16.480952,16.480952,False,rf_v1


In [78]:
features = spark.table("iceberg.gold.ml_trip_features")
features.limit(5).toPandas()

,trip_duration_minutes,total_amount,tip_amount,hour_of_day,day_of_week,week_of_year,month,year,is_weekend,is_morning,is_evening,is_night,pickup_borough,dropoff_borough,pickup_zone,dropoff_zone,is_same_borough,is_manhattan_trip,is_airport_trip,trip_distance,real_distance_miles,temp,rhum,prcp,wspd,pres,is_rainy,is_cold,is_hot,passenger_count,fare_per_mile,fare_per_minute,recent_trips_in_zone,avg_recent_duration,avg_recent_fare,VendorID,tpep_pickup_datetime
0,44.766667,42.0,0.0,7,4,40,10,2025,False,True,False,False,Brooklyn,Brooklyn,Bensonhurst West,East New York/Pennsylvania Avenue,True,False,False,11.6,6.444285,17.0,56.0,0.0,24.0,1019.0,False,True,False,1,3.620690,0.938198,100,46.531333,39.0638,1,2025-10-01 07:09:59
1,43.916667,25.0,0.0,8,4,40,10,2025,False,True,False,False,Brooklyn,Brooklyn,Bensonhurst West,Park Slope,True,False,False,4.9,4.083288,16.0,62.0,0.0,22.0,1019.0,False,True,False,1,5.102041,0.569260,100,46.859500,39.2838,1,2025-10-01 08:10:21
2,18.666667,19.0,0.0,8,4,40,10,2025,False,True,False,False,Brooklyn,Brooklyn,Bensonhurst West,Borough Park,True,False,False,2.3,1.339701,16.0,62.0,0.0,22.0,1019.0,False,True,False,1,8.260870,1.017857,100,47.002667,39.1038,1,2025-10-01 08:18:59
3,29.850000,23.0,0.0,8,4,40,10,2025,False,True,False,False,Brooklyn,Brooklyn,Bensonhurst West,Dyker Heights,True,False,False,2.6,1.098838,16.0,62.0,0.0,22.0,1019.0,False,True,False,1,8.846154,0.770519,100,46.627500,38.8338,1,2025-10-01 08:22:37
4,85.316667,47.0,0.0,8,4,40,10,2025,False,True,False,False,Brooklyn,Manhattan,Bensonhurst West,Kips Bay,False,False,False,17.4,8.913727,16.0,62.0,0.0,22.0,1019.0,False,True,False,1,2.701149,0.550889,100,46.776833,38.8738,1,2025-10-01 08:35:11


In [79]:
monitoring_df = (
    predictions.alias("p")
    .join(
        features.alias("f"),
        on="tpep_pickup_datetime",
        how="left"
    )
)


In [74]:
evaluator_rmse = RegressionEvaluator(
    labelCol="trip_duration_minutes",
    predictionCol="predicted_duration",
    metricName="rmse"
)

evaluator_mae = RegressionEvaluator(
    labelCol="trip_duration_minutes",
    predictionCol="predicted_duration",
    metricName="mae"
)

rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)

print(f"📉 RMSE: {rmse}")
print(f"📉 MAE: {mae}")


[Stage 328:====================================================>  (21 + 1) / 22]

📉 RMSE: 3.4671197087974344
📉 MAE: 1.6732009959263867


In [76]:
mape = predictions.withColumn(
    "ape",
    F.abs(F.col("trip_duration_minutes") - F.col("predicted_duration")) /
    F.col("trip_duration_minutes")
).select(F.avg("ape")).collect()[0][0]

print(f"📉 MAPE: {round(mape * 100, 2)} %")


[Stage 330:==========================================>            (17 + 2) / 22]

📉 MAPE: 11.11 %


In [80]:
drift_features = ["trip_distance", "real_distance_miles", "temp", "hour_of_day"]

drift_metrics = (
    monitoring_df
    .select(*drift_features)
    .agg(
        *[F.avg(c).alias(f"{c}_mean") for c in drift_features],
        *[F.stddev(c).alias(f"{c}_std") for c in drift_features]
    )
)

drift_metrics.show(truncate=False)


[Stage 338:=====================================================> (33 + 1) / 34]

+------------------+------------------------+-----------------+------------------+------------------+-----------------------+-----------------+-----------------+
|trip_distance_mean|real_distance_miles_mean|temp_mean        |hour_of_day_mean  |trip_distance_std |real_distance_miles_std|temp_std         |hour_of_day_std  |
+------------------+------------------------+-----------------+------------------+------------------+-----------------------+-----------------+-----------------+
|3.3844994025282733|2.4098457916775944      |14.14807594532109|14.994748338537564|4.4741410236476025|2.854572821557579      |9.234083679959504|5.236509032141719|
+------------------+------------------------+-----------------+------------------+------------------+-----------------------+-----------------+-----------------+



In [81]:
drift_metrics.limit(5).toPandas()

,trip_distance_mean,real_distance_miles_mean,temp_mean,hour_of_day_mean,trip_distance_std,real_distance_miles_std,temp_std,hour_of_day_std
0,3.384499,2.409846,14.148076,14.994748,4.474141,2.854573,9.234084,5.236509


In [84]:
pred_drift = predictions.agg(
    F.avg("predicted_duration").alias("pred_mean"),
    F.stddev("predicted_duration").alias("pred_std"),
    F.expr("percentile_approx(predicted_duration, 0.95)").alias("pred_p95")
)

pred_drift.show(truncate=False)


[Stage 352:====================================================>  (21 + 1) / 22]

+------------------+------------------+-----------------+
|pred_mean         |pred_std          |pred_p95         |
+------------------+------------------+-----------------+
|16.847838207932398|13.260995659567728|44.07562425473326|
+------------------+------------------+-----------------+



In [85]:
pred_drift_by_model = (
    predictions
    .groupBy("model_version")
    .agg(
        F.avg("predicted_duration").alias("pred_mean"),
        F.stddev("predicted_duration").alias("pred_std"),
        F.expr("percentile_approx(predicted_duration, 0.95)").alias("pred_p95"),
        F.count("*").alias("n_predictions")
    )
)

pred_drift_by_model.show(truncate=False)


[Stage 355:====================================================>  (21 + 1) / 22]

+-------------+------------------+------------------+-----------------+-------------+
|model_version|pred_mean         |pred_std          |pred_p95         |n_predictions|
+-------------+------------------+------------------+-----------------+-------------+
|rf_v1        |16.847838207932398|13.260995659567728|44.07562425473326|67510043     |
+-------------+------------------+------------------+-----------------+-------------+

